In [1]:
import pathlib
import cogent3
import madb 
from typing import Dict, List
import dataclasses
import shutil
from thesis_rec import create_thesis_rec, thesis_rec, thesis_rec_from_alignment
from time import sleep
from tqdm import tqdm

source_alignment_path = pathlib.Path('~/source/madb_data/primates100')
thesis_data = pathlib.Path('~/source/madb_data/thesis')
max_workers = 4

primates = {'gorilla': 'gorilla_gorilla', 'chimp': 'pan_troglodytes', 'macaque': 'macaca_mulatta'}


# Extract pairs of primates from alignments of all primates
 - rename sequences names to species name 
 - select only alignments that contain all 4 species 
 - select only the specific pair of species
 - remove common gaps

In [2]:

@cogent3.app.composable.define_app
def rename(align: cogent3.app.typing.AlignedSeqsType)->cogent3.app.typing.AlignedSeqsType:
    sleep(1)
    return align.rename_seqs(lambda x: x.split(':')[0])

@cogent3.app.composable.define_app
def filter_length(align: cogent3.app.typing.AlignedSeqsType, length : int = None)->cogent3.app.typing.AlignedSeqsType:
    sleep(1)
    human_sequence_length = align.get_lengths()['homo_sapiens']
    if length and human_sequence_length>length:
        return cogent3.app.composable.NotCompleted(type="FAIL",origin="filter_length",message=f"Filtered for length {human_sequence_length} > {length}", source=f'length({human_sequence_length})')
    return align

with tqdm(total=len(primates), desc="Generating primate pair alignments", unit="step") as pbar:
    for primate in primates:
        pair_name = f'human_{primate}'
        pair_path_root = thesis_data / pair_name
        pbar.set_description(f"Generating {pair_name} pair alignments")

        in_dstore = cogent3.open_data_store(source_alignment_path, suffix='fa') 
        out_dstore = cogent3.open_data_store(pair_path_root/'ensembl_alignments', suffix='fa', mode='w') 

        loader = cogent3.get_app('load_aligned', moltype='dna')
        select_4_primates = cogent3.get_app('take_named_seqs','homo_sapiens',*primates.values())
        select_pair = cogent3.get_app('take_named_seqs','homo_sapiens',primates[primate])
        omit_gaps = cogent3.get_app('omit_gap_pos', moltype="dna")
        writer = cogent3.get_app('write_seqs', data_store = out_dstore)
        app = loader + rename() + select_4_primates + select_pair + filter_length(50_000) + omit_gaps + writer 
        app.apply_to(in_dstore, show_progress=True, parallel=True, par_kw=dict(max_workers=max_workers))
        pbar.update(1)
        sleep(10)


Generating human_gorilla pair alignments:   0%|          | 0/3 [00:00<?, ?step/s]

   0%|          |00:00<?

Generating human_chimp pair alignments:  33%|███▎      | 1/3 [01:12<02:05, 62.65s/step]  

   0%|          |00:00<?

Generating human_macaque pair alignments:  67%|██████▋   | 2/3 [02:25<01:08, 68.55s/step]

   0%|          |00:00<?

Generating human_macaque pair alignments: 100%|██████████| 3/3 [03:38<00:00, 72.68s/step]


In [3]:
@cogent3.app.composable.define_app
def rec_from_alignment(aln: cogent3.app.typing.AlignedSeqsType)->cogent3.app.typing.SerialisableType:
    sleep(1)
    print(f' {aln.info.source} ', end='')
    return thesis_rec_from_alignment(aln)

@cogent3.app.composable.define_app
def cogent3_alignment(rec: cogent3.app.typing.SerialisableType)->cogent3.app.typing.SerialisableType:
    rec = thesis_rec.from_rich_dict(rec)
    sleep(1)
    return rec.align_cogent3().to_rich_dict()

@cogent3.app.composable.define_app
def sw(rec: cogent3.app.typing.SerialisableType)->cogent3.app.typing.SerialisableType:
    rec = thesis_rec.from_rich_dict(rec)
    sleep(1)
    if '' in rec.unaligned_seqs.values():
        return cogent3.NotCompleted("Empty sequence")
    if rec.unique_id == 'ENSG00000143199.fa':
        return cogent3.NotCompleted("skipping ENSG00000143199.fa")
    return rec.calc_sw().to_rich_dict()

with tqdm(total=len(primates), desc="Generating cogent3 alignments", unit="step") as pbar:
    for primate in primates:
        pair_name = f'human_{primate}'
        pair_path_root = thesis_data / pair_name
        pbar.set_description(f"Generating {pair_name} cogent3 alignments")

        in_dstore = cogent3.open_data_store(pair_path_root / 'ensembl_alignments', suffix='fa') 
        out_dstore = cogent3.open_data_store(pair_path_root / 'cogent3', suffix='json', mode='w') 

        loader = cogent3.get_app('load_aligned', moltype='dna')
        writer = cogent3.get_app('write_json', data_store=out_dstore)
        app = loader + rec_from_alignment() + cogent3_alignment() + sw() + writer 
        app.apply_to(in_dstore, show_progress=True,
                     parallel=True, par_kw=dict(max_workers=max_workers)
                     )
        if len(out_dstore.not_completed) > 0:
            print(cogent3.util.deserialise.deserialise_object(out_dstore.not_completed[0].read()))
        pbar.update(1)
        sleep(10)


 ENSG00000135776.fa  ENSG00000204518.fa  ENSG00000251246.fa  ENSG00000184389.fa 

   0%|          |00:00<?

 ENSG00000131584.fa 

/home/richard/source/madb_data/.venv/lib/python3.12/site-packages/cogent3/align/pairwise.py:1095: UserWarning: Local alignment will use > 1005.02685Mb.
  self.results[dp_options] = self.emission_probs.dp(


 ENSG00000182827.fa 

...

/home/richard/source/madb_data/.venv/lib/python3.12/site-packages/cogent3/align/pairwise.py:1095: UserWarning: Local alignment will use > 965.40868Mb.
  self.results[dp_options] = self.emission_probs.dp(


 ENSG00000173627.fa  ENSG00000143595.fa  ENSG00000143761.fa  ENSG00000186517.fa  ENSG00000132694-0.fa  ENSG00000130762.fa 

/home/richard/source/madb_data/.venv/lib/python3.12/site-packages/cogent3/align/pairwise.py:1095: UserWarning: Local alignment will use > 1284.73981Mb.
  self.results[dp_options] = self.emission_probs.dp(


 ENSG00000142632-0.fa  ENSG00000142632-1.fa 

/home/richard/source/madb_data/.venv/lib/python3.12/site-packages/cogent3/align/pairwise.py:1095: UserWarning: Local alignment will use > 2092.72483Mb.
  self.results[dp_options] = self.emission_probs.dp(
/home/richard/source/madb_data/.venv/lib/python3.12/site-packages/cogent3/align/pairwise.py:1095: UserWarning: Local alignment will use > 2524.565145Mb.
  self.results[dp_options] = self.emission_probs.dp(
/home/richard/source/madb_data/.venv/lib/python3.12/site-packages/cogent3/align/pairwise.py:1095: UserWarning: Local alignment will use > 516.02276Mb.
  self.results[dp_options] = self.emission_probs.dp(
/home/richard/source/madb_data/.venv/lib/python3.12/site-packages/cogent3/align/pairwise.py:1095: UserWarning: Local alignment will use > 3608.97444Mb.
  self.results[dp_options] = self.emission_probs.dp(
Generating human_macaque cogent3 alignments: 100%|██████████| 3/3 [37:45<00:00, 755.05s/step]


In [4]:
@cogent3.app.composable.define_app
def align_madb(rec: cogent3.app.typing.SerialisableType)->cogent3.app.typing.SerialisableType:
    rec = thesis_rec.from_rich_dict(rec)
    print(f'{rec.unique_id} ', end='')
    sleep(1)
    if rec.unique_id == 'ENSG00000162390-1.fa':
        return cogent3.NotCompleted("skipping ENSG00000162390-1.fa")
    result = rec.align_madb().to_rich_dict()
    return result

with tqdm(total=len(primates), desc="Generating ungapped smith-waterman", unit="step") as pbar:
    # primate = 'chimp'
    for primate in primates:
    # primate = 'macaque'
        pair_name = f'human_{primate}'
        pair_path_root = thesis_data / pair_name
        pbar.set_description(f"Generating {pair_name} cogent3 alignments")

        in_dstore = cogent3.open_data_store(pair_path_root / 'cogent3', suffix='json')
        out_dstore = cogent3.open_data_store(pair_path_root / 'madb', suffix='json', mode='w') 

        loader = cogent3.get_app('load_json')
        writer = cogent3.get_app('write_json', data_store = out_dstore)
        app = loader + align_madb() + writer
        app.apply_to(in_dstore, show_progress=True)
        print(out_dstore.summary_logs)
        pbar.update(1)
        sleep(10)

ENSG00000204518.fa 

   0%|          |00:00<?

ENSG00000184389.fa 

/home/richard/source/madb_data/thesis_rec.py:142: UserWarning: MADB alignment for ENSG00000184389.fa, k=10 failed: Cycle is within k(10) of the start of a bubble.   Alignment not possible
  warnings.warn(f"MADB alignment for {self.unique_id}, k={k} failed: {e}")
/home/richard/source/madb_data/thesis_rec.py:142: UserWarning: MADB alignment for ENSG00000184389.fa, k=15 failed: Cycle is within k(15) of the start of a bubble.   Alignment not possible
  warnings.warn(f"MADB alignment for {self.unique_id}, k={k} failed: {e}")
/home/richard/source/madb_data/thesis_rec.py:142: UserWarning: MADB alignment for ENSG00000184389.fa, k=20 failed: Cycle is within k(20) of the start of a bubble.   Alignment not possible
  warnings.warn(f"MADB alignment for {self.unique_id}, k={k} failed: {e}")
/home/richard/source/madb_data/thesis_rec.py:142: UserWarning: MADB alignment for ENSG00000184389.fa, k=25 failed: Cycle is within k(25) of the start of a bubble.   Alignment not possible
  warnings.warn(f"MADB 

ENSG00000131584.fa 

...

ENSG00000132694-0.fa 

/home/richard/source/madb_data/thesis_rec.py:142: UserWarning: MADB alignment for ENSG00000132694-0.fa, k=40 failed: Input sequences are not all the same length: {21001, 21004}. Please ensure all sequences are properly aligned or correct the input file format.
  warnings.warn(f"MADB alignment for {self.unique_id}, k={k} failed: {e}")


ENSG00000186517.fa ENSG00000130762.fa 

summary of log files
time                   name                                               python version    who        command                                                                                                                                                                              composable                                                                                                                                                                                                                                          
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

Generating human_macaque cogent3 alignments: 100%|██████████| 3/3 [55:53<00:00, 1117.76s/step]


In [5]:
@cogent3.app.composable.define_app
def score_alignments(rec: cogent3.app.typing.SerialisableType) -> cogent3.app.typing.SerialisableType:
    rec = thesis_rec.from_rich_dict(rec)
    rec.score_alignments().to_rich_dict()
    return rec.to_rich_dict()

primates = ['chimp', 'gorilla', 'macaque']

with tqdm(total=len(primates), desc="Scoring alignments", unit="pair") as pbar:
    for primate in primates:
        pair_name = f"human_{primate}"
        pair_path = thesis_data / pair_name 

        in_dstore = cogent3.open_data_store(pair_path / 'madb', suffix='json')
        out_dstore = cogent3.open_data_store(pair_path / 'scored', suffix='json', mode='w')

        loader = cogent3.get_app('load_json')
        writer = cogent3.get_app('write_json', data_store=out_dstore)

        app = loader + score_alignments() + writer

        app.apply_to(in_dstore, show_progress=True)

        pbar.set_description(f"Filtered {pair_name}")
        pbar.update(1)
        sleep(10)  # Optional pause between batches

   0%|          |00:00<?

   0%|          |00:00<?

   0%|          |00:00<?

Filtered human_macaque: 100%|██████████| 3/3 [00:35<00:00, 11.99s/pair]
